# w11_gpu_table1_apple_to_apple_ours_o2_full


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path
import torch

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir() and torch.cuda.is_available(), 'Kaggle GPU required'
REV = 'fce8cf24d4245d3efbc1a27ebf51790b730b86a7'
with tempfile.TemporaryDirectory(prefix='w11_t1_') as ts:
    tmp = Path(ts)
    repo = tmp / 'repo'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime-gpu', 'gdown'], check=True)
    npz_hits = list(MOUNT.rglob('paired_sample_errors.npz'))
    if npz_hits:
        paired_npz = npz_hits[0]
    else:
        z_hit = next(MOUNT.rglob('unstructured_shortpose_source_label_causal_image_results.zip'))
        paired_npz = tmp / 'paired_sample_errors.npz'
        with zipfile.ZipFile(z_hit) as zf:
            nm = next(n for n in zf.namelist() if n.endswith('paired_sample_errors.npz'))
            paired_npz.write_bytes(zf.read(nm))
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1'}
    dest = WORK / 'table1_apple_to_apple_ours_o2_full_5seed.zip'
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_table1_apple_to_apple_o2_full', str(paired_npz), str(dest), '--scratch', str(tmp / 'pinns_sc')], cwd=repo, env=env, check=True)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
